# 改善実験（評価指標拡張版）

`train_baseline.ipynb` をベースに、AUC だけでなく当たり/外れ構造も見ながら改善案を比較する。

- ベース前処理: `lib.get_baseline_data()`
- 弱点セグメント: `review_year=2015`, `genre_Documentary=1`, `top_critic=True`
- 出力: 実験結果CSV、セグメント集計CSV、提出ファイル


In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from lib import get_baseline_data, BASELINE_FEATURES, run_full_analysis

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

OUTPUT_DIR = "outputs"
os.makedirs(os.path.join(OUTPUT_DIR, "submissions"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "experiments"), exist_ok=True)
print("Setup complete.")


Setup complete.


In [2]:
train, test = get_baseline_data()
BASE_FEATURES = BASELINE_FEATURES.copy()
print(f"Train: {len(train):,}, Test: {len(test):,}, BaseFeatures: {len(BASE_FEATURES)}")

VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))
print(f"TimeCV folds: {len(time_splits)} (years={VAL_YEARS})")


Train: 653,507, Test: 40,716, BaseFeatures: 38
TimeCV folds: 4 (years=[2013, 2014, 2015, 2016])


In [3]:
lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
    "random_state": 42, "verbosity": -1,
}

def _add_config_features(tr_df, val_df, te_df, config_name):
    extra = []
    if config_name == "base":
        return extra

    if config_name in ["publisher_freq", "publisher_freq__doc_x_critic_te"]:
        vc = tr_df["publisher_name"].astype(str).value_counts()
        tr_df["publisher_name_freq"] = tr_df["publisher_name"].astype(str).map(vc).fillna(0).astype(float)
        val_df["publisher_name_freq"] = val_df["publisher_name"].astype(str).map(vc).fillna(0).astype(float)
        te_df["publisher_name_freq"] = te_df["publisher_name"].astype(str).map(vc).fillna(0).astype(float)
        extra.append("publisher_name_freq")

    if config_name in ["doc_x_critic_te", "publisher_freq__doc_x_critic_te"]:
        tr_df["doc_x_critic_te"] = tr_df["genre_Documentary"] * tr_df["critic_name_te_ts"]
        val_df["doc_x_critic_te"] = val_df["genre_Documentary"] * val_df["critic_name_te_ts"]
        te_df["doc_x_critic_te"] = te_df["genre_Documentary"] * te_df["critic_name_te_ts"]
        extra.append("doc_x_critic_te")

    if config_name == "year_norm_x_critic_te":
        y_min = tr_df["review_year"].min()
        y_max = tr_df["review_year"].max()
        den = max(y_max - y_min, 1)
        tr_df["review_year_norm_x_critic_te"] = ((tr_df["review_year"] - y_min) / den) * tr_df["critic_name_te_ts"]
        val_df["review_year_norm_x_critic_te"] = ((val_df["review_year"] - y_min) / den) * val_df["critic_name_te_ts"]
        te_df["review_year_norm_x_critic_te"] = ((te_df["review_year"] - y_min) / den) * te_df["critic_name_te_ts"]
        extra.append("review_year_norm_x_critic_te")

    if config_name == "topcritic_x_critic_te":
        tr_df["topcritic_x_critic_te"] = tr_df["top_critic"].astype(int) * tr_df["critic_name_te_ts"]
        val_df["topcritic_x_critic_te"] = val_df["top_critic"].astype(int) * val_df["critic_name_te_ts"]
        te_df["topcritic_x_critic_te"] = te_df["top_critic"].astype(int) * te_df["critic_name_te_ts"]
        extra.append("topcritic_x_critic_te")

    return extra

def evaluate_config(config_name):
    y_all = train["target"].values
    oof = np.full(len(train), np.nan)
    fold_rows = []

    for i, (tr_idx, val_idx) in enumerate(time_splits):
        tr_df = train.iloc[tr_idx].copy()
        val_df = train.iloc[val_idx].copy()
        te_df = test.copy()
        extra = _add_config_features(tr_df, val_df, te_df, config_name)
        feats = BASE_FEATURES + extra

        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(tr_df[feats], tr_df["target"].values,
                  eval_set=[(val_df[feats], val_df["target"].values)],
                  callbacks=[lgb.early_stopping(20, verbose=False)])

        val_pred = model.predict_proba(val_df[feats])[:, 1]
        oof[val_idx] = val_pred

        y_val = val_df["target"].values
        pred_label = (val_pred >= 0.5).astype(int)
        tn = int(((y_val == 0) & (pred_label == 0)).sum())
        fp = int(((y_val == 0) & (pred_label == 1)).sum())
        tp = int(((y_val == 1) & (pred_label == 1)).sum())
        fn = int(((y_val == 1) & (pred_label == 0)).sum())

        fold_rows.append({
            "fold": i + 1,
            "val_year": VAL_YEARS[i],
            "auc": roc_auc_score(y_val, val_pred),
            "logloss": log_loss(y_val, val_pred),
            "brier": brier_score_loss(y_val, val_pred),
            "fp_rate": fp / max(tn + fp, 1),
            "fn_rate": fn / max(tp + fn, 1),
            "pred_pos_rate": pred_label.mean(),
            "pos_rate": y_val.mean(),
        })

    val_mask = ~np.isnan(oof)
    val_df_all = train.loc[val_mask].copy()
    y_val_all = train["target"].values[val_mask]
    oof_val = oof[val_mask]
    _, summaries = run_full_analysis(
        val_df_all, y_val_all, oof_val,
        group_cols=["review_year", "genre_Documentary", "top_critic"],
        min_count=100,
    )

    def _seg_auc(summary_df, col_name, key):
        if summary_df is None or summary_df.empty:
            return np.nan
        hit = summary_df[summary_df[col_name] == key]
        if len(hit) == 0:
            return np.nan
        return float(hit.iloc[0]["auc"])

    s_year = summaries.get("review_year", pd.DataFrame())
    s_doc = summaries.get("genre_Documentary", pd.DataFrame())
    s_top = summaries.get("top_critic", pd.DataFrame())

    fold_df = pd.DataFrame(fold_rows)
    result = {
        "config": config_name,
        "n_feat": len(BASE_FEATURES) + (0 if config_name == "base" else len(_add_config_features(train.iloc[:200].copy(), train.iloc[:200].copy(), test.iloc[:200].copy(), config_name))),
        "auc_mean": fold_df["auc"].mean(),
        "auc_std": fold_df["auc"].std(),
        "logloss_mean": fold_df["logloss"].mean(),
        "brier_mean": fold_df["brier"].mean(),
        "fp_rate_mean": fold_df["fp_rate"].mean(),
        "fn_rate_mean": fold_df["fn_rate"].mean(),
        "pos_gap_mean": (fold_df["pred_pos_rate"] - fold_df["pos_rate"]).mean(),
        "auc_review_2015": _seg_auc(s_year, "review_year", 2015),
        "auc_doc_1": _seg_auc(s_doc, "genre_Documentary", 1),
        "auc_topcritic_true": _seg_auc(s_top, "top_critic", True),
    }
    return result, fold_df


In [4]:
EXPERIMENT_CONFIGS = [
    "base",
    "publisher_freq",
    "doc_x_critic_te",
    "year_norm_x_critic_te",
    "topcritic_x_critic_te",
    "publisher_freq__doc_x_critic_te",
]

results = []
fold_details = []
for cfg in EXPERIMENT_CONFIGS:
    row, fold_df = evaluate_config(cfg)
    results.append(row)
    fold_df["config"] = cfg
    fold_details.append(fold_df)
    print(f"{cfg}: auc={row['auc_mean']:.4f} +- {row['auc_std']:.4f}, doc_auc={row['auc_doc_1']:.4f}")

result_df = pd.DataFrame(results).sort_values("auc_mean", ascending=False).reset_index(drop=True)
fold_detail_df = pd.concat(fold_details, axis=0, ignore_index=True)

result_df.to_csv(os.path.join(OUTPUT_DIR, "experiments", "metric_driven_experiment_results.csv"), index=False)
fold_detail_df.to_csv(os.path.join(OUTPUT_DIR, "experiments", "metric_driven_fold_details.csv"), index=False)

print(f"\nSaved: {OUTPUT_DIR}/experiments/metric_driven_experiment_results.csv")
print(f"Saved: {OUTPUT_DIR}/experiments/metric_driven_fold_details.csv")
display(result_df)


base: auc=0.7602 +- 0.0064, doc_auc=0.7149
publisher_freq: auc=0.7601 +- 0.0070, doc_auc=0.7148
doc_x_critic_te: auc=0.7612 +- 0.0056, doc_auc=0.7179
year_norm_x_critic_te: auc=0.7608 +- 0.0058, doc_auc=0.7150
topcritic_x_critic_te: auc=0.7602 +- 0.0064, doc_auc=0.7149
publisher_freq__doc_x_critic_te: auc=0.7616 +- 0.0058, doc_auc=0.7177

Saved: metric_driven_experiment_results.csv
Saved: metric_driven_fold_details.csv


,config,n_feat,auc_mean,auc_std,logloss_mean,brier_mean,fp_rate_mean,fn_rate_mean,pos_gap_mean,auc_review_2015,auc_doc_1,auc_topcritic_true
0,publisher_freq__doc_x_critic_te,40,0.761568,0.005794,0.543142,0.183040,0.554858,0.133256,0.105639,0.753982,0.717655,0.748221
1,doc_x_critic_te,39,0.761211,0.005642,0.543443,0.183147,0.554416,0.133185,0.105536,0.753834,0.717936,0.747722
2,year_norm_x_critic_te,39,0.760839,0.005803,0.543701,0.183243,0.560760,0.130039,0.109791,0.752849,0.715000,0.746896
3,base,38,0.760220,0.006443,0.544179,0.183457,0.555654,0.133508,0.105768,0.751233,0.714856,0.746741
4,topcritic_x_critic_te,39,0.760220,0.006443,0.544179,0.183457,0.555654,0.133508,0.105768,0.751233,0.714856,0.746741
5,publisher_freq,39,0.760117,0.007007,0.544194,0.183452,0.556351,0.132578,0.106618,0.750244,0.714828,0.746641


In [13]:
# 提出作成（選んだ config で全train学習）
SELECTED_CONFIG = "year_norm_x_critic_te"  # 例: "publisher_freq", "doc_x_critic_te"

train_full = train.copy()
test_full = test.copy()
extra = _add_config_features(train_full, test_full, test_full, SELECTED_CONFIG)
feats = BASE_FEATURES + extra

model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(train_full[feats], train_full["target"].values)
pred = model_full.predict_proba(test_full[feats])[:, 1]

out_name = f"submission_{SELECTED_CONFIG}.csv"
pd.DataFrame({"ID": test_full["ID"], "target": pred}).to_csv(out_name, index=False)
print(f"Saved {out_name} (n_feat={len(feats)})")

importance_df = pd.DataFrame({"feature": feats, "importance": model_full.feature_importances_}).sort_values("importance", ascending=False)
display(importance_df.head(20))


Saved submission_year_norm_x_critic_te.csv (n_feat=39)


,feature,importance
7,directors,741
8,authors,561
0,rotten_tomatoes_link,493
4,movie_title,407
1,critic_name,216
3,publisher_name,180
9,actors,77
11,production_company,74
5,movie_info,53
28,production_company_te_ts,43


In [9]:
train["target"]

0         1
1         1
2         0
3         0
4         1
         ..
653502    1
653503    1
653504    1
653505    1
653506    0
Name: target, Length: 653507, dtype: int64

In [11]:
train.head()

,ID,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_date,movie_title,movie_info,content_rating,genres,...,production_company_te_ts,critic_name_te_ts_bin,production_company_te_ts_bin,runtime_bin,movie_age_bin,release_decade,movie_info_len,movie_info_word_count,movie_title_len,movie_title_word_count
0,0,m/0814255,Andrew L. Urban,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489462,2.0,1.0,1.0,0,2010.0,454,79,50,8
1,1,m/0814255,Louise Keller,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489491,2.0,1.0,1.0,0,2010.0,454,79,50,8
2,2,m/0814255,David Germain,True,Associated Press,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489582,1.0,1.0,1.0,0,2010.0,454,79,50,8
3,3,m/0814255,Nick Schager,False,Slant Magazine,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489637,1.0,1.0,1.0,0,2010.0,454,79,50,8
4,4,m/0814255,Bill Goodykoontz,True,Arizona Republic,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489549,2.0,1.0,1.0,0,2010.0,454,79,50,8
